# Phase 5: Algorithm Exploration for Centerline Generation

## 5A — Evaluation Harness & Baselines
## 5B — Data Characterization

**Objective**: Build a quantitative evaluation framework, establish baselines,
and deeply characterize VPD/HPD/Nav Streets data to inform algorithm selection.

**Reference**: Nav Streets is the existing road network (not ground truth — our
generated centerlines should **exceed** its coverage). VPD is the high-confidence
anchor per hackathon rules §4.1.

In [7]:
import sys, os, time, warnings, importlib
warnings.filterwarnings('ignore')

# Ensure project root is on path (notebook is in notebooks/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
# If running from project root already, adjust
if not os.path.isdir(os.path.join(PROJECT_ROOT, 'src')):
    PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from shapely.geometry import box, LineString, Point
from shapely.ops import unary_union
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)-25s %(levelname)-5s %(message)s')
logger = logging.getLogger('notebook')

# Force-reload evaluation module to pick up any code changes
import src.evaluation.metrics as _metrics_mod
importlib.reload(_metrics_mod)

import src.config
# UPDATE: Use the full Nav Streets overlap region instead of the small tile
src.config.BBOX = (21.089, 42.571, 21.142, 42.626)
from src.config import BBOX, CRS, OUTPUTS_DIR

# Reload loaders to ensure they pick up the updated BBOX
import src.loaders.nav_loader
import src.loaders.vpd_loader
import src.loaders.hpd_loader
importlib.reload(src.loaders.nav_loader)
importlib.reload(src.loaders.vpd_loader)
importlib.reload(src.loaders.hpd_loader)

from src.loaders import load_vpd, load_hpd, load_nav_streets
from src.preprocessing.cleaning import compute_heading, densify_gdf, validate_geometries
from src.evaluation.metrics import (
    evaluate_centerlines, CenterlineMetrics, print_metrics, metrics_to_dict,
    nav_recovery_and_precision, smoothness_metrics, topology_metrics, METRIC_CRS
)

FIGURES_DIR = OUTPUTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'BBOX: {BBOX}')
print(f'Metric CRS: {METRIC_CRS}')

Working directory: e:\Coding projects\HERE-Geospatial-Hackathon
Project root: e:\Coding projects\HERE-Geospatial-Hackathon
BBOX: (21.088588, 42.571255, 21.188588, 42.671255)
Metric CRS: EPSG:32634


## Load All Datasets

In [3]:
t0 = time.time()

print('Loading Nav Streets...')
nav_gdf = load_nav_streets()
print(f'  Nav Streets: {len(nav_gdf)} links')

print('Loading HPD traces...')
hpd_gdf = load_hpd()
print(f'  HPD: {len(hpd_gdf)} traces')

print('Loading VPD (this takes a while)...')
vpd_gdf = load_vpd()
print(f'  VPD: {len(vpd_gdf)} rows')

print(f'\nAll loaded in {time.time()-t0:.1f}s')

2026-02-10 17:55:19,616 src.loaders.nav_loader    INFO  Loading Nav Streets from E:\Coding projects\HERE-Geospatial-Hackathon\data\Kosovo_s_nav_streets\Kosovo.gpkg…


Loading Nav Streets...


2026-02-10 17:55:19,999 src.loaders.nav_loader    INFO  Bbox clip: 4353 → 1250 road links.
2026-02-10 17:55:20,000 src.loaders.nav_loader    INFO  Nav Streets loaded: 1250 road links, 14 columns.
2026-02-10 17:55:20,001 src.loaders.hpd_loader    INFO  Loading HPD week 1 from E:\Coding projects\HERE-Geospatial-Hackathon\data\Kosovo_HPD\XKO_HPD_week_1.csv…


  Nav Streets: 1250 links
Loading HPD traces...


2026-02-10 17:55:20,291 src.loaders.hpd_loader    INFO  Loading HPD week 2 from E:\Coding projects\HERE-Geospatial-Hackathon\data\Kosovo_HPD\XKO_HPD_week_2.csv…
2026-02-10 17:55:20,603 src.loaders.hpd_loader    INFO  Combined HPD: 613733 probe points.
2026-02-10 17:55:20,701 src.loaders.hpd_loader    INFO  Reconstructing traces from 613733 probe points…
2026-02-10 17:55:23,004 src.loaders.hpd_loader    INFO  Reconstructed 5872 traces.
2026-02-10 17:55:23,031 src.loaders.hpd_loader    INFO  Bbox clip: 5872 → 5517 traces.
2026-02-10 17:55:23,033 src.loaders.hpd_loader    INFO  HPD loaded: 5517 traces.
2026-02-10 17:55:23,052 src.loaders.vpd_loader    INFO  Loading VPD from E:\Coding projects\HERE-Geospatial-Hackathon\data\Kosovo_VPD\Kosovo_VPD.csv (chunk_size=10000)…


  HPD: 5517 traces
Loading VPD (this takes a while)...


2026-02-10 17:55:51,269 src.loaders.vpd_loader    INFO  Read 54542 rows total, kept 54542 fused rows.
2026-02-10 17:55:51,353 src.loaders.vpd_loader    INFO  Parsing WKT geometries…
2026-02-10 18:04:34,492 src.loaders.vpd_loader    INFO  Bbox clip: 54542 → 54542 rows.
2026-02-10 18:04:34,527 src.loaders.vpd_loader    INFO  VPD loaded: 54542 rows, 13 columns.


  VPD: 54542 rows

All loaded in 557.7s


---
# PHASE 5A — Evaluation Harness & Baselines

We establish two baselines:
1. **Nav Streets self-score** — perfect recovery (100%), zero new roads. This is the floor.
2. **Raw VPD as-is** — no processing, just the raw fused trajectories treated as centerlines. High coverage but massive redundancy and poor topology.

### Baseline 1: Nav Streets (Self-Evaluation)

In [8]:
# Force fresh reference to reloaded functions
from src.evaluation.metrics import evaluate_centerlines, print_metrics, metrics_to_dict

# Nav Streets scored against itself — the trivial baseline
print('Evaluating Nav Streets vs Nav Streets (self-score)...')
nav_baseline = evaluate_centerlines(
    generated=nav_gdf,
    nav=nav_gdf,
    buffer_m=15.0,
    skip_redundancy=False,
)
print_metrics(nav_baseline, label='Nav Streets (Self-Score)')

2026-02-10 18:18:22,295 src.evaluation.metrics    INFO  Evaluating centerlines: 1250 generated vs 1250 Nav Streets
2026-02-10 18:18:22,369 src.evaluation.metrics    INFO    Computing recovery & precision (buffer=15m)...


Evaluating Nav Streets vs Nav Streets (self-score)...


2026-02-10 18:18:57,407 src.evaluation.metrics    INFO    Computing per-link Hausdorff distances...
2026-02-10 18:18:58,162 src.evaluation.metrics    INFO    Computing smoothness metrics...
2026-02-10 18:18:58,206 src.evaluation.metrics    INFO    Computing topology metrics...
2026-02-10 18:18:58,324 src.evaluation.metrics    INFO    Computing redundancy (buffer=5m)...
2026-02-10 18:18:58,683 src.evaluation.metrics    INFO    Evaluation complete.



  Evaluation: Nav Streets (Self-Score)
  Metric                                          Value
  ---------------------------------------- ------------
  nav_recovery_pct                              100.000
  nav_precision_pct                             100.000
  new_road_km                                     0.000
  mean_hausdorff_m                              275.420
  median_hausdorff_m                            182.100
  mean_angular_deflection_deg                   168.930
  median_angular_deflection_deg                 175.788
  total_length_km                               161.661
  num_lines                                        1250
  num_components                                      9
  num_dangling_ends                                 209
  num_pseudo_nodes                                  250
  num_intersections                                 318
  redundant_length_km                            10.702
  redundancy_pct                                  6.620



### Baseline 2: Raw VPD (No Processing)

Treat every VPD fused trajectory as a "centerline". This gives us the upper bound on
coverage (we have all the data) but worst-case redundancy and topology.

In [ ]:
# Raw VPD baseline — use smaller sample and compute metrics individually
# (full evaluate_centerlines is too slow for 5K raw trajectories)
from src.evaluation.metrics import (
    nav_recovery_and_precision, smoothness_metrics, topology_metrics, _to_metric, METRIC_CRS
)

VPD_SAMPLE_N = 2000
vpd_sample = vpd_gdf.sample(min(VPD_SAMPLE_N, len(vpd_gdf)), random_state=42)

print(f'Computing raw VPD baseline ({len(vpd_sample)} trajectories)...')
print('Step 1/3: Recovery & precision...')
ref = nav_recovery_and_precision(vpd_sample, nav_gdf, buffer_m=15.0)
print(f'  Nav recovery: {ref["nav_recovery_pct"]}%')
print(f'  Nav precision: {ref["nav_precision_pct"]}%')
print(f'  New road potential: {ref["new_road_km"]:.1f} km')

print('Step 2/3: Smoothness...')
sm = smoothness_metrics(vpd_sample)
print(f'  Mean angular deflection: {sm["mean_angular_deflection_deg"]:.2f}°')

print('Step 3/3: Topology...')
topo = topology_metrics(vpd_sample, snap_tolerance=1.0)
print(f'  Components: {topo["num_components"]}')
print(f'  Dangling ends: {topo["num_dangling_ends"]}')
print(f'  Intersections: {topo["num_intersections"]}')

gen_proj = _to_metric(vpd_sample)
total_km = gen_proj.geometry.length.sum() / 1000.0

# Build a CenterlineMetrics manually
vpd_baseline = CenterlineMetrics(
    nav_recovery_pct=ref["nav_recovery_pct"],
    nav_precision_pct=ref["nav_precision_pct"],
    new_road_km=ref["new_road_km"],
    mean_hausdorff_m=0.0,  # skip for raw VPD (too expensive, not meaningful)
    median_hausdorff_m=0.0,
    mean_angular_deflection_deg=sm["mean_angular_deflection_deg"],
    median_angular_deflection_deg=sm["median_angular_deflection_deg"],
    total_length_km=round(total_km, 3),
    num_lines=len(vpd_sample),
    num_components=topo["num_components"],
    num_dangling_ends=topo["num_dangling_ends"],
    num_pseudo_nodes=topo["num_pseudo_nodes"],
    num_intersections=topo["num_intersections"],
    redundant_length_km=0.0,  # skip
    redundancy_pct=0.0,
)
print_metrics(vpd_baseline, label=f'Raw VPD ({len(vpd_sample)} sample)')

Computing raw VPD baseline (2000 trajectories)...
Step 1/3: Recovery & precision...


### Baseline Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Nav Streets (self)': metrics_to_dict(nav_baseline),
    f'Raw VPD ({len(vpd_sample)} sample)': metrics_to_dict(vpd_baseline),
}).T

print('\n=== Baseline Comparison ===')
display(comparison.style.format(precision=3))

---
# PHASE 5B — Data Characterization

Deep analysis of VPD, HPD, and Nav Streets spatial properties to
inform which clustering and centerline extraction algorithm will work best.

## 5B.1 — Spatial Density Heatmap

Rasterize VPD + HPD coordinate points onto a 10m grid to see where data is dense
(easy clustering) vs sparse (hard / gap-filling needed).

In [ ]:
# Project to metric CRS
vpd_proj = vpd_gdf.to_crs(METRIC_CRS)
hpd_proj = hpd_gdf.to_crs(METRIC_CRS)
nav_proj = nav_gdf.to_crs(METRIC_CRS)

# Extract all coordinate points from VPD and HPD LineStrings
def extract_points(gdf, sample_n=None):
    """Extract all vertex coordinates from LineStrings in a GeoDataFrame."""
    if sample_n and len(gdf) > sample_n:
        gdf = gdf.sample(sample_n, random_state=42)
    xs, ys = [], []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        if hasattr(geom, 'geoms'):  # MultiLineString
            for part in geom.geoms:
                coords = list(part.coords)
                xs.extend([c[0] for c in coords])
                ys.extend([c[1] for c in coords])
        else:
            coords = list(geom.coords)
            xs.extend([c[0] for c in coords])
            ys.extend([c[1] for c in coords])
    return np.array(xs), np.array(ys)

print('Extracting VPD points...')
vpd_xs, vpd_ys = extract_points(vpd_proj)
print(f'  VPD: {len(vpd_xs):,} coordinate points')

print('Extracting HPD points...')
hpd_xs, hpd_ys = extract_points(hpd_proj)
print(f'  HPD: {len(hpd_xs):,} coordinate points')

In [ ]:
# Build density heatmaps on a 10m grid
GRID_SIZE = 10  # metres

# Get bounds from projected bbox
bbox_proj = gpd.GeoDataFrame(geometry=[box(*BBOX)], crs=CRS).to_crs(METRIC_CRS)
bxmin, bymin, bxmax, bymax = bbox_proj.total_bounds

x_edges = np.arange(bxmin, bxmax + GRID_SIZE, GRID_SIZE)
y_edges = np.arange(bymin, bymax + GRID_SIZE, GRID_SIZE)

print(f'Grid: {len(x_edges)-1} x {len(y_edges)-1} cells ({GRID_SIZE}m resolution)')

vpd_density, _, _ = np.histogram2d(vpd_xs, vpd_ys, bins=[x_edges, y_edges])
hpd_density, _, _ = np.histogram2d(hpd_xs, hpd_ys, bins=[x_edges, y_edges])
combined_density = vpd_density + hpd_density

print(f'VPD: max density = {vpd_density.max():.0f} pts/cell, non-zero cells = {(vpd_density > 0).sum():,}')
print(f'HPD: max density = {hpd_density.max():.0f} pts/cell, non-zero cells = {(hpd_density > 0).sum():,}')
print(f'Combined: non-zero cells = {(combined_density > 0).sum():,}')

In [ ]:
# Plot density heatmaps
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, data, title in zip(
    axes,
    [vpd_density.T, hpd_density.T, combined_density.T],
    ['VPD Point Density', 'HPD Point Density', 'Combined VPD+HPD']
):
    # Use log scale for better visualization
    data_plot = np.where(data > 0, np.log10(data), np.nan)
    im = ax.imshow(
        data_plot, origin='lower', aspect='equal',
        extent=[bxmin, bxmax, bymin, bymax],
        cmap='hot_r', interpolation='nearest'
    )
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    plt.colorbar(im, ax=ax, label='log10(point count)', shrink=0.7)

plt.suptitle(f'Spatial Density Heatmaps ({GRID_SIZE}m grid)', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'phase5b_density_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: phase5b_density_heatmap.png')

## 5B.2 — Trajectory Overlap Analysis

Sample VPD trajectories and compute pairwise Hausdorff distances to understand
natural clustering thresholds. This tells us what `eps` value for DBSCAN-like
clustering is appropriate.

In [ ]:
# Sample 300 VPD trajectories for pairwise analysis (300x300 = 90K pairs)
N_SAMPLE = 300
vpd_sample_traj = vpd_proj.sample(min(N_SAMPLE, len(vpd_proj)), random_state=42).reset_index(drop=True)

print(f'Computing pairwise Hausdorff distances for {len(vpd_sample_traj)} trajectories...')

# Compute pairwise directed Hausdorff distances
n = len(vpd_sample_traj)
hausdorff_matrix = np.zeros((n, n))

geoms = vpd_sample_traj.geometry.values
for i in range(n):
    for j in range(i+1, n):
        try:
            hd = geoms[i].hausdorff_distance(geoms[j])
            hausdorff_matrix[i, j] = hd
            hausdorff_matrix[j, i] = hd
        except Exception:
            hausdorff_matrix[i, j] = np.inf
            hausdorff_matrix[j, i] = np.inf

# Extract upper triangle (unique pairs)
upper_tri = hausdorff_matrix[np.triu_indices(n, k=1)]
upper_tri_finite = upper_tri[np.isfinite(upper_tri)]

print(f'Pairwise distances computed: {len(upper_tri_finite):,} pairs')
print(f'  Min: {upper_tri_finite.min():.1f}m')
print(f'  Median: {np.median(upper_tri_finite):.1f}m')
print(f'  Mean: {np.mean(upper_tri_finite):.1f}m')
print(f'  Percentiles: 5th={np.percentile(upper_tri_finite, 5):.1f}m, '
      f'25th={np.percentile(upper_tri_finite, 25):.1f}m, '
      f'75th={np.percentile(upper_tri_finite, 75):.1f}m, '
      f'95th={np.percentile(upper_tri_finite, 95):.1f}m')

In [ ]:
# Histogram of pairwise Hausdorff distances — zoom into the 0-100m range
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
axes[0].hist(upper_tri_finite, bins=100, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Pairwise Hausdorff Distances (Full)')
axes[0].set_xlabel('Hausdorff Distance (m)')
axes[0].set_ylabel('Count')
axes[0].axvline(np.median(upper_tri_finite), color='red', linestyle='--', label=f'Median={np.median(upper_tri_finite):.0f}m')
axes[0].legend()

# Zoomed: 0-100m — this is where clustering thresholds live
close_pairs = upper_tri_finite[upper_tri_finite < 100]
axes[1].hist(close_pairs, bins=50, color='darkorange', edgecolor='black', alpha=0.7)
axes[1].set_title(f'Close Pairs (<100m): {len(close_pairs):,} / {len(upper_tri_finite):,}')
axes[1].set_xlabel('Hausdorff Distance (m)')
axes[1].set_ylabel('Count')

# Mark candidate clustering thresholds
for thresh in [5, 10, 15, 20]:
    n_below = (upper_tri_finite < thresh).sum()
    axes[1].axvline(thresh, color='red', linestyle=':', alpha=0.6)
    axes[1].text(thresh+0.5, axes[1].get_ylim()[1]*0.9, f'{thresh}m\n({n_below})',
                fontsize=8, color='red')

plt.suptitle('Trajectory Overlap Analysis — Clustering Threshold Selection', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'phase5b_hausdorff_distribution.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: phase5b_hausdorff_distribution.png')

## 5B.3 — Heading Distribution per Grid Cell

Compute heading of each VPD segment and map heading distributions spatially.
Bimodal heading = two-way divided road (needs 2 centerlines). Unimodal = one-way or narrow.

In [ ]:
# Compute heading for each VPD trajectory
print('Computing headings for VPD trajectories...')
vpd_proj_copy = vpd_proj.copy()
vpd_proj_copy['heading'] = vpd_proj_copy.geometry.apply(compute_heading)

# Extract centroid + heading for each trajectory
vpd_proj_copy['cx'] = vpd_proj_copy.geometry.centroid.x
vpd_proj_copy['cy'] = vpd_proj_copy.geometry.centroid.y

# Assign each trajectory to a grid cell (50m grid for heading analysis)
HEADING_GRID = 50  # metres
vpd_proj_copy['gx'] = ((vpd_proj_copy['cx'] - bxmin) / HEADING_GRID).astype(int)
vpd_proj_copy['gy'] = ((vpd_proj_copy['cy'] - bymin) / HEADING_GRID).astype(int)
vpd_proj_copy['cell'] = vpd_proj_copy['gx'].astype(str) + '_' + vpd_proj_copy['gy'].astype(str)

print(f'Grid cells with data: {vpd_proj_copy["cell"].nunique()}')
print(f'Heading stats: mean={vpd_proj_copy["heading"].mean():.1f}°, '
      f'std={vpd_proj_copy["heading"].std():.1f}°')

In [ ]:
# For each grid cell, compute heading statistics and detect bimodality
from scipy import stats

def heading_modality(headings, n_bins=36):
    """
    Detect if heading distribution is bimodal (two-way road) or unimodal.
    Returns: 'bimodal', 'unimodal', or 'sparse' (too few samples).
    Also returns the dominant heading(s).
    """
    if len(headings) < 5:
        return 'sparse', []
    
    # Bin headings into 10-degree bins
    bins = np.arange(0, 370, 10)
    hist, _ = np.histogram(headings, bins=bins)
    
    # Find peaks (bins with counts > 20% of max)
    threshold = hist.max() * 0.3
    peak_bins = np.where(hist > threshold)[0]
    
    if len(peak_bins) == 0:
        return 'sparse', []
    
    # Group consecutive peak bins
    groups = []
    current = [peak_bins[0]]
    for i in range(1, len(peak_bins)):
        # Handle circular (0 and 35 are adjacent)
        if peak_bins[i] - peak_bins[i-1] <= 2:
            current.append(peak_bins[i])
        else:
            groups.append(current)
            current = [peak_bins[i]]
    groups.append(current)
    
    # Check if first and last groups wrap around 0/360
    if len(groups) > 1 and (groups[-1][-1] >= 34 and groups[0][0] <= 1):
        groups[0] = groups[-1] + groups[0]
        groups.pop()
    
    dominant_headings = [bins[int(np.mean(g))] for g in groups]
    
    if len(groups) >= 2:
        return 'bimodal', dominant_headings
    else:
        return 'unimodal', dominant_headings

# Analyze each cell
cell_modality = {}
for cell_id, group in vpd_proj_copy.groupby('cell'):
    modality, dom_headings = heading_modality(group['heading'].values)
    gx, gy = cell_id.split('_')
    cell_modality[cell_id] = {
        'gx': int(gx), 'gy': int(gy),
        'modality': modality,
        'count': len(group),
        'dominant_headings': dom_headings
    }

mod_df = pd.DataFrame(cell_modality).T
mod_counts = mod_df['modality'].value_counts()
print('Heading modality per grid cell:')
print(mod_counts)
print(f'\n→ {mod_counts.get("bimodal", 0)} cells have bimodal heading (likely divided two-way roads)')
print(f'→ {mod_counts.get("unimodal", 0)} cells have unimodal heading (one-way or narrow roads)')

In [ ]:
# Visualize heading modality map
fig, ax = plt.subplots(figsize=(12, 10))

# Plot Nav Streets as background
nav_proj.plot(ax=ax, color='lightgray', linewidth=0.5, alpha=0.5, label='Nav Streets')

# Color each cell by modality
modality_colors = {'bimodal': 'red', 'unimodal': 'blue', 'sparse': 'gray'}

for _, row in mod_df.iterrows():
    gx, gy = int(row['gx']), int(row['gy'])
    x0 = bxmin + gx * HEADING_GRID
    y0 = bymin + gy * HEADING_GRID
    rect = plt.Rectangle((x0, y0), HEADING_GRID, HEADING_GRID,
                         color=modality_colors[row['modality']],
                         alpha=0.4)
    ax.add_patch(rect)

legend_elements = [
    Patch(facecolor='red', alpha=0.4, label=f'Bimodal ({mod_counts.get("bimodal", 0)} cells)'),
    Patch(facecolor='blue', alpha=0.4, label=f'Unimodal ({mod_counts.get("unimodal", 0)} cells)'),
    Patch(facecolor='gray', alpha=0.4, label=f'Sparse ({mod_counts.get("sparse", 0)} cells)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)
ax.set_title(f'Heading Modality Map ({HEADING_GRID}m grid)\nRed=Divided Road | Blue=One-way/Narrow', fontsize=13)
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.set_xlim(bxmin, bxmax)
ax.set_ylim(bymin, bymax)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'phase5b_heading_modality.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: phase5b_heading_modality.png')

## 5B.4 — VPD vs HPD Spatial Complementarity

Which grid cells have VPD only, HPD only, or both? This tells us where HPD adds value
as a gap-filler vs. where it's just noise on already-covered roads.

In [ ]:
# Build binary coverage grids (50m cells)
COMP_GRID = 50  # metres
comp_x_edges = np.arange(bxmin, bxmax + COMP_GRID, COMP_GRID)
comp_y_edges = np.arange(bymin, bymax + COMP_GRID, COMP_GRID)

vpd_comp, _, _ = np.histogram2d(vpd_xs, vpd_ys, bins=[comp_x_edges, comp_y_edges])
hpd_comp, _, _ = np.histogram2d(hpd_xs, hpd_ys, bins=[comp_x_edges, comp_y_edges])

vpd_mask = vpd_comp > 0
hpd_mask = hpd_comp > 0

both_mask = vpd_mask & hpd_mask
vpd_only_mask = vpd_mask & ~hpd_mask
hpd_only_mask = ~vpd_mask & hpd_mask
neither_mask = ~vpd_mask & ~hpd_mask

total_cells = vpd_mask.size
print(f'Grid: {vpd_comp.shape[0]}x{vpd_comp.shape[1]} = {total_cells:,} cells ({COMP_GRID}m)')
print(f'  Both VPD+HPD:  {both_mask.sum():>6,} cells ({both_mask.sum()/total_cells*100:.1f}%)')
print(f'  VPD only:      {vpd_only_mask.sum():>6,} cells ({vpd_only_mask.sum()/total_cells*100:.1f}%)')
print(f'  HPD only:      {hpd_only_mask.sum():>6,} cells ({hpd_only_mask.sum()/total_cells*100:.1f}%) ← gap-filler value')
print(f'  Neither:       {neither_mask.sum():>6,} cells ({neither_mask.sum()/total_cells*100:.1f}%)')

In [ ]:
# Complementarity map
comp_grid = np.zeros_like(vpd_comp, dtype=int)
comp_grid[both_mask] = 3       # Both
comp_grid[vpd_only_mask] = 2   # VPD only
comp_grid[hpd_only_mask] = 1   # HPD only
# 0 = neither

fig, ax = plt.subplots(figsize=(12, 10))

cmap = mcolors.ListedColormap(['white', 'orange', 'blue', 'green'])
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

im = ax.imshow(comp_grid.T, origin='lower', aspect='equal',
               extent=[bxmin, bxmax, bymin, bymax],
               cmap=cmap, norm=norm, interpolation='nearest')

# Nav Streets overlay
nav_proj.plot(ax=ax, color='black', linewidth=0.5, alpha=0.3)

legend_elements = [
    Patch(facecolor='green', label=f'Both VPD+HPD ({both_mask.sum()} cells)'),
    Patch(facecolor='blue', label=f'VPD only ({vpd_only_mask.sum()} cells)'),
    Patch(facecolor='orange', label=f'HPD only ({hpd_only_mask.sum()} cells)'),
    Patch(facecolor='white', edgecolor='gray', label=f'No data ({neither_mask.sum()} cells)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)
ax.set_title(f'VPD vs HPD Spatial Complementarity ({COMP_GRID}m grid)', fontsize=13)
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'phase5b_vpd_hpd_complementarity.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: phase5b_vpd_hpd_complementarity.png')

## 5B.5 — Z-Level / Altitude Analysis

Inspect VPD `altitudes` to understand multi-level road occurrences
(bridges, overpasses, tunnels). Determines if altitude-based trace separation is needed.

In [ ]:
# Parse altitudes — vpd_gdf should have 'altitudes' column as parsed lists
print('Analyzing VPD altitude data...')

# Check what altitudes look like
sample_alts = vpd_gdf['altitudes'].dropna().head(10)
print('Sample altitudes (first 10 rows):')
for i, alt in enumerate(sample_alts):
    if isinstance(alt, (list, np.ndarray)):
        print(f'  Row {i}: {len(alt)} values, range=[{min(alt):.1f}, {max(alt):.1f}]')
    else:
        print(f'  Row {i}: {type(alt).__name__} = {str(alt)[:80]}')

In [ ]:
# Compute mean altitude per trajectory
def safe_mean_alt(alt):
    if isinstance(alt, (list, np.ndarray)) and len(alt) > 0:
        vals = [v for v in alt if v is not None and not np.isnan(v)]
        return np.mean(vals) if vals else np.nan
    return np.nan

def safe_std_alt(alt):
    if isinstance(alt, (list, np.ndarray)) and len(alt) > 1:
        vals = [v for v in alt if v is not None and not np.isnan(v)]
        return np.std(vals) if len(vals) > 1 else 0.0
    return np.nan

vpd_gdf_alt = vpd_gdf.copy()
vpd_gdf_alt['mean_alt'] = vpd_gdf_alt['altitudes'].apply(safe_mean_alt)
vpd_gdf_alt['std_alt'] = vpd_gdf_alt['altitudes'].apply(safe_std_alt)

valid_alts = vpd_gdf_alt['mean_alt'].dropna()
print(f'Trajectories with altitude data: {len(valid_alts):,} / {len(vpd_gdf):,} '
      f'({len(valid_alts)/len(vpd_gdf)*100:.1f}%)')

if len(valid_alts) > 0:
    print(f'Altitude statistics:')
    print(f'  Min:    {valid_alts.min():.1f}m')
    print(f'  Max:    {valid_alts.max():.1f}m')
    print(f'  Mean:   {valid_alts.mean():.1f}m')
    print(f'  Std:    {valid_alts.std():.1f}m')
    print(f'  Median: {valid_alts.median():.1f}m')
    
    # Detect significant altitude variations (potential multi-level)
    high_var = vpd_gdf_alt[vpd_gdf_alt['std_alt'] > 5]  # >5m variation within a single trajectory
    print(f'\nTrajectories with >5m altitude variation (potential multi-level): '
          f'{len(high_var)} ({len(high_var)/len(vpd_gdf)*100:.2f}%)')

In [ ]:
# Spatial map of altitude + detect multi-level areas
if len(valid_alts) > 100:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Plot mean altitude per trajectory (centroid)
    vpd_alt_proj = vpd_gdf_alt.dropna(subset=['mean_alt']).to_crs(METRIC_CRS)
    vpd_alt_proj['cx'] = vpd_alt_proj.geometry.centroid.x
    vpd_alt_proj['cy'] = vpd_alt_proj.geometry.centroid.y
    
    # Sample for plotting speed
    plot_sample = vpd_alt_proj.sample(min(10000, len(vpd_alt_proj)), random_state=42)
    
    sc1 = axes[0].scatter(plot_sample['cx'], plot_sample['cy'],
                          c=plot_sample['mean_alt'], cmap='terrain',
                          s=1, alpha=0.3)
    plt.colorbar(sc1, ax=axes[0], label='Mean Altitude (m)', shrink=0.7)
    axes[0].set_title('Mean Altitude per Trajectory')
    axes[0].set_aspect('equal')
    
    # Plot altitude standard deviation (high std = potential multi-level area)
    sc2 = axes[1].scatter(plot_sample['cx'], plot_sample['cy'],
                          c=plot_sample['std_alt'].clip(0, 10), cmap='Reds',
                          s=1, alpha=0.3)
    plt.colorbar(sc2, ax=axes[1], label='Altitude Std Dev (m)', shrink=0.7)
    axes[1].set_title('Altitude Variation (high = multi-level?)')
    axes[1].set_aspect('equal')
    
    for ax in axes:
        ax.set_xlabel('Easting (m)')
        ax.set_ylabel('Northing (m)')
    
    plt.suptitle('VPD Altitude Analysis', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'phase5b_altitude_analysis.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved: phase5b_altitude_analysis.png')
else:
    print('Insufficient altitude data for spatial analysis.')

In [ ]:
# Detect multi-level conflict zones:
# Grid cells where trajectories at same location have >8m altitude difference
if len(valid_alts) > 100:
    vpd_alt_proj_full = vpd_gdf_alt.dropna(subset=['mean_alt']).to_crs(METRIC_CRS).copy()
    vpd_alt_proj_full['cx'] = vpd_alt_proj_full.geometry.centroid.x
    vpd_alt_proj_full['cy'] = vpd_alt_proj_full.geometry.centroid.y
    vpd_alt_proj_full['agx'] = ((vpd_alt_proj_full['cx'] - bxmin) / 50).astype(int)
    vpd_alt_proj_full['agy'] = ((vpd_alt_proj_full['cy'] - bymin) / 50).astype(int)
    vpd_alt_proj_full['acell'] = vpd_alt_proj_full['agx'].astype(str) + '_' + vpd_alt_proj_full['agy'].astype(str)
    
    # Per cell: altitude range
    cell_alt_range = vpd_alt_proj_full.groupby('acell')['mean_alt'].agg(['min', 'max', 'count', 'std'])
    cell_alt_range['range'] = cell_alt_range['max'] - cell_alt_range['min']
    
    # Cells with >8m altitude range AND >5 trajectories (not just noise)
    multilevel = cell_alt_range[(cell_alt_range['range'] > 8) & (cell_alt_range['count'] > 5)]
    
    print(f'Multi-level conflict zones (>8m altitude range, >5 trajectories): {len(multilevel)}')
    if len(multilevel) > 0:
        print(multilevel.sort_values('range', ascending=False).head(10))
    else:
        print('No significant multi-level conflicts detected. Z-level separation may not be critical.')

## 5B.6 — Nav Streets Coverage Gap Map

Highlight areas where VPD traces exist but no nearby Nav Street link.
These are the "new road" discovery opportunities.

In [ ]:
# Build Nav Streets buffer (15m) and identify VPD cells outside it
NAV_BUFFER_M = 15.0

print(f'Building Nav Streets buffer ({NAV_BUFFER_M}m)...')
nav_union_buf = nav_proj.geometry.buffer(NAV_BUFFER_M).unary_union

# For each VPD trajectory centroid: is it inside the Nav buffer?
vpd_centroids = vpd_proj.copy()
vpd_centroids['centroid_geom'] = vpd_centroids.geometry.centroid

print('Checking VPD centroids against Nav buffer...')
vpd_centroids['near_nav'] = vpd_centroids['centroid_geom'].apply(
    lambda pt: nav_union_buf.contains(pt) if pt and not pt.is_empty else False
)

near_count = vpd_centroids['near_nav'].sum()
gap_count = len(vpd_centroids) - near_count
print(f'VPD trajectories near Nav Streets: {near_count:,} ({near_count/len(vpd_centroids)*100:.1f}%)')
print(f'VPD trajectories in GAP areas:     {gap_count:,} ({gap_count/len(vpd_centroids)*100:.1f}%) ← new road potential')

In [ ]:
# Gap map visualization
fig, ax = plt.subplots(figsize=(14, 12))

# Nav Streets
nav_proj.plot(ax=ax, color='black', linewidth=1.5, alpha=0.8, label='Nav Streets', zorder=3)

# VPD near Nav (known roads) — light blue
vpd_near = vpd_centroids[vpd_centroids['near_nav']]
vpd_gap = vpd_centroids[~vpd_centroids['near_nav']]

if len(vpd_near) > 10000:
    vpd_near_plot = vpd_near.sample(10000, random_state=42)
else:
    vpd_near_plot = vpd_near

if len(vpd_gap) > 10000:
    vpd_gap_plot = vpd_gap.sample(10000, random_state=42)
else:
    vpd_gap_plot = vpd_gap

vpd_near_plot.plot(ax=ax, color='lightblue', linewidth=0.3, alpha=0.2, label='VPD (on known roads)', zorder=1)
vpd_gap_plot.plot(ax=ax, color='red', linewidth=0.5, alpha=0.5, label=f'VPD (GAP = new roads) [{gap_count:,}]', zorder=2)

ax.legend(loc='upper right', fontsize=11)
ax.set_title(f'Nav Streets Coverage Gaps — New Road Discovery Potential\n'
             f'{gap_count:,} VPD trajectories ({gap_count/len(vpd_centroids)*100:.1f}%) '
             f'outside {NAV_BUFFER_M}m Nav buffer', fontsize=13)
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'phase5b_nav_coverage_gaps.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: phase5b_nav_coverage_gaps.png')

---
## Summary of Findings

Run the cell below after all analyses to get a consolidated summary.

In [ ]:
print('=' * 70)
print('  PHASE 5A+5B SUMMARY')
print('=' * 70)

print('\n--- 5A: Evaluation Baselines ---')
print(f'Nav Streets self-score:')
print(f'  Recovery: {nav_baseline.nav_recovery_pct}%  |  Smoothness: {nav_baseline.mean_angular_deflection_deg}°')
print(f'  Topology: {nav_baseline.num_components} components, {nav_baseline.num_dangling_ends} dangling, {nav_baseline.num_intersections} intersections')
print(f'\nRaw VPD baseline ({len(vpd_sample)} sample):')
print(f'  Recovery: {vpd_baseline.nav_recovery_pct}%  |  Smoothness: {vpd_baseline.mean_angular_deflection_deg}°')
print(f'  Topology: {vpd_baseline.num_components} components, {vpd_baseline.num_dangling_ends} dangling, {vpd_baseline.num_intersections} intersections')
print(f'  New road: {vpd_baseline.new_road_km:.1f} km')

print('\n--- 5B: Data Characterization ---')
print(f'VPD total coord points: {len(vpd_xs):,}')
print(f'HPD total coord points: {len(hpd_xs):,}')
print(f'Density grid ({GRID_SIZE}m): VPD non-zero={int((vpd_density>0).sum()):,}, HPD non-zero={int((hpd_density>0).sum()):,}')
print(f'Heading modality: bimodal={mod_counts.get("bimodal", 0)}, unimodal={mod_counts.get("unimodal", 0)}, sparse={mod_counts.get("sparse", 0)}')
print(f'Complementarity ({COMP_GRID}m): HPD-only cells={int(hpd_only_mask.sum())} (gap-filler value)')
print(f'Nav gaps: {gap_count:,} VPD trajectories ({gap_count/len(vpd_centroids)*100:.1f}%) outside Nav buffer')

print('\n--- Key Takeaways for Algorithm Selection ---')
print('1. Heading-aware clustering IS needed (bimodal cells indicate divided roads)')
print('2. HPD gap-fill value quantified — focus HPD integration on HPD-only cells')
print('3. Nav recovery baseline established — algorithms must beat raw VPD score')
print('4. Multi-level analysis shows whether Z-level separation is critical')
print('5. Natural clustering threshold visible from Hausdorff distribution')
print('=' * 70)